# Data Preprocessing

In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

import joblib

In [4]:
# load dataset
df = pd.read_csv(
    "../data/atlas-higgs-challenge-2014-v2.csv.gz"
)

print(df.shape)
df.head()

(818238, 35)


,EventId,DER_mass_MMC,DER_mass_transverse_met_lep,DER_mass_vis,DER_pt_h,DER_deltaeta_jet_jet,DER_mass_jet_jet,DER_prodeta_jet_jet,DER_deltar_tau_lep,DER_pt_tot,...,PRI_jet_leading_eta,PRI_jet_leading_phi,PRI_jet_subleading_pt,PRI_jet_subleading_eta,PRI_jet_subleading_phi,PRI_jet_all_pt,Weight,Label,KaggleSet,KaggleWeight
0,100000,138.470,51.655,97.827,27.980,0.91,124.711,2.666,3.064,41.928,...,2.150,0.444,46.062,1.24,-2.475,113.497,0.000814,s,t,0.002653
1,100001,160.937,68.768,103.235,48.146,-999.00,-999.000,-999.000,3.473,2.078,...,0.725,1.158,-999.000,-999.00,-999.000,46.226,0.681042,b,t,2.233584
2,100002,-999.000,162.172,125.953,35.635,-999.00,-999.000,-999.000,3.148,9.336,...,2.053,-2.028,-999.000,-999.00,-999.000,44.251,0.715742,b,t,2.347389
3,100003,143.905,81.417,80.943,0.414,-999.00,-999.000,-999.000,3.310,0.414,...,-999.000,-999.000,-999.000,-999.00,-999.000,-0.000,1.660654,b,t,5.446378
4,100004,175.864,16.915,134.805,16.405,-999.00,-999.000,-999.000,3.891,16.405,...,-999.000,-999.000,-999.000,-999.00,-999.000,0.000,1.904263,b,t,6.245333


### 1.Drop Unnecessary Columns

In [5]:
# drop unnecessary columns
df = df.drop(columns=['EventId', 'Weight', 'KaggleSet', 'KaggleWeight'])

### 2.Encode Target Labels

In [6]:
# convert label to binary numeric
df["Label"] = df["Label"].map({
    "b": 0,
    "s": 1
})

In [7]:
df["Label"].value_counts()

Label
0    538678
1    279560
Name: count, dtype: int64

### 3.Separate Features and Target

In [8]:
X = df.drop(columns=["Label"])
y = df["Label"]

# Hnandling missing values

In [9]:
# replace -999 with NaN for imputation
X = X.replace(-999, np.nan)

In [10]:
# check missing values
X.isnull().sum().sort_values(
    ascending=False
).head(10)

DER_deltaeta_jet_jet      580253
DER_mass_jet_jet          580253
DER_lep_eta_centrality    580253
DER_prodeta_jet_jet       580253
PRI_jet_subleading_pt     580253
PRI_jet_subleading_phi    580253
PRI_jet_subleading_eta    580253
PRI_jet_leading_pt        327371
PRI_jet_leading_eta       327371
PRI_jet_leading_phi       327371
dtype: int64

### Train / Validation / Test Split

In [11]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

In [12]:
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    random_state=42,
    stratify=y_temp
)

In [13]:
print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(572766, 30)
(122736, 30)
(122736, 30)


### Impute Missing Values

In [14]:
# impute missing values using median strategy
imputer = SimpleImputer(
    strategy="median"
)

X_train = imputer.fit_transform(X_train)

X_val = imputer.transform(X_val)

X_test = imputer.transform(X_test)

In [15]:
joblib.dump(
    imputer,
    "../models/imputer.pkl"
)

['../models/imputer.pkl']

# Feature Scaling

In [16]:
# scale features using StandardScaler
scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)

X_val = scaler.transform(X_val)

X_test = scaler.transform(X_test)


joblib.dump(
    scaler,
    "../models/scaler.pkl"
)

['../models/scaler.pkl']

## Convert Back to DataFrame


In [17]:
# save feature names for later use
feature_names = X.columns

In [18]:
# create a DataFrame for the training set to visualize the preprocessed data 
train_df = pd.DataFrame(
    X_train,
    columns=feature_names
)

train_df["Label"] = y_train.values

In [19]:
# create a DataFrame for the validation set to visualize the preprocessed data
val_df = pd.DataFrame(
    X_val,
    columns=feature_names
)

val_df["Label"] = y_val.values

In [20]:
# create a DataFrame for the test set to visualize the preprocessed data
test_df = pd.DataFrame(
    X_test,
    columns=feature_names
)

test_df["Label"] = y_test.values

In [21]:
train_df.to_csv(
    "../data/processed/train.csv",
    index=False
)

val_df.to_csv(
    "../data/processed/validation.csv",
    index=False
)

test_df.to_csv(
    "../data/processed/test.csv",
    index=False
)

## Preprocessing Summary

1. Removed EventId column.
2. Converted labels:
   - Signal (s) → 1
   - Background (b) → 0
3. Replaced -999 values with NaN.
4. Imputed missing values using median.
5. Performed stratified train-validation-test split.
6. Standardized features using StandardScaler.
7. Saved processed datasets for downstream modeling.